In [ ]:
import pandas as pd
import numpy as np
from collections import deque
import random

def backtest_ticks(
    csv_path,
    init_usdt=1000.0,
    init_tmn=100_000_000.0,
    fee=0.001,
    K=2.0,
    ROLLING=True,
    ROLL_N=20,
    STATIC_LOWER=-100.0,
    STATIC_UPPER=100.0,
    ORDER_QTY_USDT=100.0,
    mode="taker",       # "taker" | "mid" | "maker"
    p_fill=1.0,         # fill probability for mid/maker orders
    tick_size=1.0       # one tick (e.g. 1 TMN)
):
    """
    Backtest USDT↔TMN arbitrage strategy on historical tick data.

    Strategy logic:
    1. Input tick data includes y_bid, y_ask, y_mid (Wallex) and x_mid (Nobitex).
       Spread variable is defined as:
           z = y_mid - x_mid
    2. Rolling bands are calculated:
           upper = mean(z) + K * std(z)
           lower = mean(z) - K * std(z)
       If rolling mode is disabled or not enough samples, static bands are used.
    3. Trade signals:
       - If z > upper → SELL USDT (receive TMN)
       - If z < lower → BUY USDT (pay TMN)
    4. Trade execution modes:
       - "taker": always executed at best bid/ask (100% fill)
       - "mid": order placed at (bid+ask)/2 with fill probability = p_fill
       - "maker": order placed one tick better than best bid/ask
                  (buy at bid+tick, sell at ask-tick) with fill probability = p_fill
    5. Fees:
       - Always applied for any mode.
       - SELL: received TMN = qty * px * (1 - fee)
       - BUY:  paid TMN = qty * px * (1 + fee)
    6. Portfolio tracking:
       - Balances of USDT and TMN are updated after each fill.
       - If insufficient balance, order is skipped.
    7. Benchmark:
       - All portfolio converted into USDT at the first tick
       - Final value compared with strategy performance
       - Report includes both strategy and benchmark results

    Returns:
        trades_df (pd.DataFrame): log of all executed trades
        report (dict): summary performance metrics
    """

    # Load dataset
    df = pd.read_csv(csv_path)

    # Portfolio initial values
    first_mid = df.iloc[0]["y_mid"]
    last_mid  = df.iloc[-1]["y_mid"]

    usdt = init_usdt
    tmn = init_tmn

    zs = deque(maxlen=ROLL_N if ROLLING else 1)
    trades = []

    for i, row in df.iterrows():
        ts = row["ts_utc"]
        w_bid, w_ask, w_mid = row["y_bid"], row["y_ask"], row["y_mid"]
        x_mid = row["x_mid"]
        z = w_mid - x_mid

        # --- Calculate trading bands
        if ROLLING:
            zs.append(z)

        have_roll = (ROLLING and len(zs) == ROLL_N)
        if have_roll:
            mean = np.mean(zs)
            std = np.std(zs, ddof=1) if len(zs) > 1 else 0.0
            upper, lower = mean + K * std, mean - K * std
            mode_band = "rolling"
        else:
            upper, lower = STATIC_UPPER, STATIC_LOWER
            mode_band = "static"

        # ============================
        #   SELL SIGNAL
        # ============================
        if z > upper:
            if usdt >= ORDER_QTY_USDT:
                qty = ORDER_QTY_USDT

                # Select execution price
                if mode == "taker":
                    px = w_bid
                elif mode == "mid":
                    px = (w_bid + w_ask) / 2
                elif mode == "maker":
                    px = w_ask - tick_size
                else:
                    raise ValueError("Invalid mode")

                # Check fill probability
                if mode == "taker" or random.random() < p_fill:
                    tmn_gain = qty * px * (1 - fee)
                    usdt -= qty
                    tmn += tmn_gain
                    trades.append({
                        "ts": ts, "side": "SELL", "price": px, "qty": qty,
                        "z": z, "usdt": usdt, "tmn": tmn, "band_mode": mode_band,
                        "exec_mode": mode
                    })

        # ============================
        #   BUY SIGNAL
        # ============================
        elif z < lower:
            qty = ORDER_QTY_USDT

            if mode == "taker":
                px = w_ask
            elif mode == "mid":
                px = (w_bid + w_ask) / 2
            elif mode == "maker":
                px = w_bid + tick_size
            else:
                raise ValueError("Invalid mode")

            cost = qty * px * (1 + fee)
            if tmn >= cost:
                if mode == "taker" or random.random() < p_fill:
                    tmn -= cost
                    usdt += qty
                    trades.append({
                        "ts": ts, "side": "BUY", "price": px, "qty": qty,
                        "z": z, "usdt": usdt, "tmn": tmn, "band_mode": mode_band,
                        "exec_mode": mode
                    })

    # Convert trades to DataFrame
    trades_df = pd.DataFrame(trades)

    # Final portfolio value
    final_value_strategy = tmn + usdt * last_mid
    init_value = init_tmn + init_usdt * first_mid
    pnl_strategy = final_value_strategy - init_value
    ret_pct_strategy = pnl_strategy / init_value * 100

    # Benchmark: all converted to USDT at start
    init_usdt_benchmark = init_usdt + init_tmn / first_mid
    final_value_benchmark = init_usdt_benchmark * last_mid
    pnl_benchmark = final_value_benchmark - init_value
    ret_pct_benchmark = pnl_benchmark / init_value * 100

    relative = (final_value_strategy / final_value_benchmark - 1) * 100

    report = {
        "init_value": init_value,
        "final_value_strategy": final_value_strategy,
        "pnl_strategy": pnl_strategy,
        "ret_pct_strategy": ret_pct_strategy,
        "final_value_benchmark": final_value_benchmark,
        "pnl_benchmark": pnl_benchmark,
        "ret_pct_benchmark": ret_pct_benchmark,
        "relative_outperformance_pct": relative,
        "trades_count": len(trades_df)
    }

    return trades_df, report


In [ ]:
import os
import pandas as pd
import itertools

def grid_search_backtest(
    csv_path,
    output_dir=None,
    windows=[100, 200],
    ks=[1.0, 2.0],
    modes=["taker", "mid", "maker"],   # <── new input: which modes to run
    ps_mid=[0.5],
    ps_maker=[0.3],
    n_rep=3,
    init_usdt=1000,
    init_tmn=100_000_000,
    fee=0.001,
    order_qty=100
):
    """
    Grid search backtest for USDT↔TMN arbitrage.

    Parameters:
        csv_path (str): path to tick data CSV
        output_dir (str): directory to save results (default: same as csv_path)
        windows (list): list of rolling window sizes
        ks (list): list of K multipliers
        modes (list): which modes to include ["taker","mid","maker"]
        ps_mid (list): fill probabilities for mid mode
        ps_maker (list): fill probabilities for maker mode
        n_rep (int): number of replications for probabilistic modes
        init_usdt, init_tmn (float): initial balances
        fee (float): trading fee
        order_qty (float): order size (in USDT)

    Output:
        results_raw.csv  → every replication as a row
        results_mean.csv → mean results grouped by (window,K,mode,p_fill)

    Returns:
        df_raw, df_mean (pd.DataFrame)
    """

    # default output_dir = same folder as csv_path
    if output_dir is None:
        output_dir = os.path.dirname(csv_path)

    raw_path = os.path.join(output_dir, "results_raw.csv")
    mean_path = os.path.join(output_dir, "results_mean.csv")

    # remove old files if exist
    for f in [raw_path, mean_path]:
        if os.path.exists(f):
            os.remove(f)

    all_results = []

    # iterate over windows, ks, modes
    for win, k, mode in itertools.product(windows, ks, modes):

        # set prob list and rep count depending on mode
        if mode == "taker":
            p_list = [1.0]   # always fill
            reps = 1
        elif mode == "mid":
            p_list = ps_mid
            reps = n_rep
        elif mode == "maker":
            p_list = ps_maker
            reps = n_rep
        else:
            raise ValueError(f"Invalid mode: {mode}")

        for p in p_list:
            for rep in range(reps):
                trades_df, report = backtest_ticks(
                    csv_path=csv_path,
                    init_usdt=init_usdt,
                    init_tmn=init_tmn,
                    fee=fee,
                    K=k,
                    ROLLING=True,
                    ROLL_N=win,
                    STATIC_LOWER=-250,
                    STATIC_UPPER=250,
                    ORDER_QTY_USDT=order_qty,
                    mode=mode,
                    p_fill=p,
                    tick_size=1.0
                )

                # count buys/sells
                buy_count = (trades_df["side"] == "BUY").sum()
                sell_count = (trades_df["side"] == "SELL").sum()

                row = {
                    "window": win,
                    "K": k,
                    "mode": mode,
                    "p_fill": p,
                    "rep_id": rep,
                    "trades_count": report["trades_count"],
                    "growth_strategy": report["final_value_strategy"] / report["init_value"],
                    "growth_bench": report["final_value_benchmark"] / report["init_value"],
                    "rel_perf": (report["final_value_strategy"] / report["final_value_benchmark"]),
                    "buy_count": buy_count,
                    "sell_count": sell_count
                }

                all_results.append(row)

                # incremental save
                pd.DataFrame([row]).to_csv(
                    raw_path, mode="a", header=not os.path.exists(raw_path), index=False
                )

    # after loop: compute mean results
    df_raw = pd.DataFrame(all_results)
    df_mean = df_raw.groupby(["window", "K", "mode", "p_fill"]).mean(numeric_only=True).reset_index()
    df_mean.to_csv(mean_path, index=False)

    return df_raw, df_mean


In [ ]:
df_raw, df_mean = grid_search_backtest(
    csv_path='data_path',
    windows=[720*i for i in range(1,4)],
    ks=[1 + i*5 for i in range(0,4)],
    modes=["taker"],   # <── only taker and maker
    ps_mid=[],
    ps_maker=[],
    n_rep=1,
    init_usdt=1000,
    init_tmn=0,
    fee=0.001,
    order_qty=120
)
